In [1]:
import torch

In [5]:
# 题目1：逐行归一化（L2归一化）
# 给定一个 shape 为 (64, 10) 的 tensor x（模拟模型输出 logit），实现“每行元素的 L2 归一化”，即每行所有元素的平方和开根号后，用每行元素除以该根号值，最终每行的 L2 范数为 1。
#提示：用 torch.norm 或 torch.sqrt + torch.sum（注意 dim 和 keepdim 的使用，避免广播错误）
x = torch.randn(64, 10)

# L2范数：
l2 = torch.linalg.norm(x, dim=1, keepdim=True)
x_norm = x / l2

In [ ]:
# 题目2：批量生成 one-hot 编码
# 给定一个 shape 为 (32,) 的标签 tensor label（每个元素取值 0~9，模拟 10 分类标签），
# 生成对应的 one-hot 编码，输出 shape 为 (32, 10)，要求每行只有对应标签位置为 1，其余为 0。
#提示：用 torch.zeros_like + torch.scatter_，结合 torch.unsqueeze 扩展维度（参考你之前写的 one-hot 代码，简化实现）。

x = torch.randint(0, 10, size=(32,))
# one-hot
one_hot = torch.zeros(size=(32, 10))
y = torch.arange(0, 32)
one_hot[(y,x)]= 1
one_hot


torch.Size([1, 32, 10])

In [13]:
# 题目3：筛选并替换元素
# 给定一个 shape 为 (100,) 的随机 tensor x（torch.randn(100)），实现两个操作：
# ① 将所有小于 0 的元素替换为 0；② 将所有大于 2 的元素替换为 2（模拟 ReLU6 激活函数的效果）。
# 提示：用 torch.where 或 torch.clamp（推荐 clamp，更简洁，贴合激活函数场景）

x = torch.randn(100,)
index1 = x<0
index2 = x>0.5
x[index1] = 0
x[index2] = 2
x

tensor([2.0000, 2.0000, 0.0000, 0.0000, 0.0000, 2.0000, 2.0000, 0.0000, 0.1739,
        2.0000, 0.0000, 0.0000, 2.0000, 0.0000, 2.0000, 2.0000, 2.0000, 2.0000,
        0.0000, 2.0000, 0.0000, 2.0000, 0.0000, 2.0000, 0.0000, 0.0000, 0.0000,
        0.0167, 0.0000, 2.0000, 0.2751, 0.0000, 0.0355, 2.0000, 0.0000, 0.0000,
        2.0000, 0.4311, 0.0000, 0.0000, 0.0000, 0.3847, 0.0000, 2.0000, 0.0576,
        0.0000, 0.0000, 0.0000, 0.4594, 2.0000, 0.0000, 0.0000, 2.0000, 2.0000,
        2.0000, 0.0000, 2.0000, 2.0000, 2.0000, 0.0000, 0.0000, 2.0000, 0.0000,
        0.0000, 0.0717, 0.4947, 0.1059, 0.0000, 2.0000, 2.0000, 2.0000, 0.0000,
        0.0000, 2.0000, 2.0000, 2.0000, 2.0000, 0.2717, 0.0000, 2.0000, 0.0000,
        0.0000, 0.0000, 0.1682, 0.0000, 2.0000, 0.0000, 0.1778, 2.0000, 0.4926,
        0.0000, 0.0000, 0.0000, 0.3992, 2.0000, 0.1633, 0.0000, 0.0000, 2.0000,
        0.0000])

In [ ]:
# 题目7：索引提取与重组
# 给定一个 shape 为 (64, 10) 的 tensor x，以及一个 shape 为 (64,) 的索引 tensor idx（每个元素取值 0~9），
# 要求提取 x 中“每行对应 idx 索引位置的元素”，输出 shape 为 (64,)，然后将该 tensor 扩展为 (64, 1)，并与原 x 拼接，最终输出 shape 为 (64, 11)。
# 提示：用 torch.gather 或 torch.index_select 提取元素，结合 torch.unsqueeze、torch.cat 实现拼接（避免循环逐个提取）。

x = torch.randn(64, 10)
idx = torch.randint(0, 10, size=(64, ))
y = x[(torch.arange(0,64), idx)].unsqueeze(1) # 也可用torch.gather(x, dim=1, index=idx.unsqueeze(1))
final = torch.cat((x, y), dim=1)


torch.Size([64, 11])

In [ ]:
# 题目8：多标签交叉熵的向量化实现
# 给定多标签标签 tensor label（shape (32, 10)，元素为 0 或 1，一个样本可属于多个类别），
# 以及模型输出 logit（shape (32, 10)），用向量化操作实现多标签交叉熵损失（均值），
# 公式：loss = -mean( label×log(sigmoid(logit)) + (1-label)×log(1-sigmoid(logit)) )，要求加入 1e-7 防止 log(0)。
# 提示：结合 torch.sigmoid、torch.log、torch.mul，利用广播机制实现逐元素计算，无需循环遍历每个类别。
label = torch.randint(0, 2, size=(32, 10))
y = torch.randn(32, 10)
# # softmax
# logit = torch.exp(y) / torch.exp(y).sum(dim=1, keepdim=True)
# sigmoid
logit = torch.exp(-y) / (1 + torch.exp(-y).sum(dim=1, keepdim=True))
logit = logit + 1e-7 # 防止log0

# 计算cross_entropy可以直接用logx_target, 也可以用矩阵逐点相乘
loss = - torch.mean( label * torch.log(logit) + (1 - label) * torch.log(1 - logit)) 



tensor(1.5572)